In [ ]:
"""
Author: Sophie A. Liu
Date: 06/04/2026 4:38pm
Purpose: NSF to isolate genes using Townes & Engelhardt (2022)
"""

In [ ]:
import squidpy as sq
import scanpy as sc

from matplotlib import pyplot as plt
import tqdm as notebook_tqdm
import pandas as pd
import numpy as np

# tensorflow is incompatible with later versions of python, must force 3.11.
from tensorflow_probability import math as tm
tfk = tm.psd_kernels

# pulled from github repo
from models import sf
from utils import misc,preprocess,training,postprocess,visualize

In [ ]:
# df = pd.read_parquet("ICsquare_016um/spatial/tissue_positions.parquet")
# df.to_csv("tissue_position_list.csv", index=False)

In [ ]:
pd1_vis = sq.read.visium("square_016um", load_images=True)         # 16 micron is wide but need fewer obs for processing
pd1_vis.var_names_make_unique()

iso_vis = sq.read.visium("ICsquare_016um", load_images=True) 
iso_vis.var_names_make_unique()

In [6]:
# needing coarser bins or subsetting area if I have to. regionally or random sampling 
space = pd1_vis.obsm["spatial"]
xlim = (5000, 9000)        # coords pixels not microns anymore
ylim = (5000, 8000)

x = space[:, 0]
y = space[:, 1]

box = (xlim[0] < x) & (x < xlim[1]) & (ylim[0] < y) & (y < ylim[1])

pd1_cropped = pd1_vis[box, :]
space = pd1_cropped.obsm["spatial"]

# same for iso
space2 = iso_vis.obsm["spatial"]
xlim2 = (2000, 6000)       # boxes of equal area
ylim2 = (2000, 5000)

x2 = space2[:, 0]
y2 = space2[:, 1]

box2 = (xlim2[0] < x2) & (x2 < xlim2[1]) & (ylim2[0] < y2) & (y2 < ylim2[1])

iso_cropped = iso_vis[box2, :]
space2 = iso_cropped.obsm["spatial"]

In [7]:
import anndata as ad
merged = ad.concat([pd1_cropped, iso_cropped], join = "outer")

In [9]:
# ideally enough info but taking out noise
sc.pp.highly_variable_genes(
    merged, 
    n_top_genes=4000, 
    flavor="seurat_v3"        # for raw counts data
)

keep = merged.var["highly_variable"].values

hvg = merged[:, keep]

In [10]:
# initializing vars and conds
D,Dval = preprocess.anndata_to_train_val(hvg, layer=None, train_frac=0.8, 
                                         flip_yaxis=False)      
Ntr,J = D["Y"].shape
Xtr = D["X"]

# pd1_cropped = pd1_cropped[:Ntr,:]
hvg = hvg[:Ntr,:]

# tensors for mathing yippee
Dtf = preprocess.prepare_datasets_tf(D,Dval=Dval)

In [ ]:
# inducing points, gaussian process ML needs optimization "jumping points"
L = 9                                     # base on ideal NMF factor rank but generous Bayesian shrinkage 
Z = misc.kmeans_inducing_pts(Xtr, 500)    # KMeans can be a bit crude
M = Z.shape[0]
ker = tfk.MaternThreeHalves

In [ ]:
# diagnostics and making sure capabilities of matrix are fine.
print("Y:", D["Y"].shape)
print("Xtr:", Xtr.shape)
print("Z:", Z.shape)
print("J:", J)
print("L:", L)

In [ ]:
# training the model to find NSFs
# our data fits NB more but it won't converge. latent factor fine gau and downstream specify family in modeling but I am not 100% happy
fit = sf.SpatialFactorization(J, L, Z, psd_kernel=ker, nonneg=True, lik="gau")      
fit.init_loadings(D["Y"], X=Xtr, sz=D["sz"], shrinkage=0.3)                        
tro = training.ModelTrainer(fit)

%time tro.train_model(*Dtf, status_freq=50)                               
visualize.plot_loss(tro.loss)        # diminishing returns past 300 epochs

In [ ]:
# postprocessing and visualization/verification of spatial overlay
insf = postprocess.interpret_nsf(fit,Xtr,S=100,lda_mode=False)        # S number of neighbors

In [ ]:
# using model for prediction, smoothes out (points not in training set)
insf2 = postprocess.interpret_nsf(fit,space,S=100,lda_mode=False)

In [ ]:
coords = pd.DataFrame(hvg.obsm["spatial"], columns=["x", "y"])

In [ ]:
factors = pd.DataFrame(
    insf["factors"],
    index=coords.index,
    columns=[f"NSF_{i+1}" for i in range(insf["factors"].shape[1])]
)

In [ ]:
joined = pd.concat([coords, factors], axis=1)
joined.head()

In [ ]:
# getting the files for GSEA, have to extract separately :(
tops = fit.W.numpy()
gene_names = hvg.var_names
factor_names = [f"{i+1}" for i in range(tops.shape[1])]

geneWeights = pd.DataFrame(tops, index=gene_names, columns=factor_names)
geneWeights.head()

In [ ]:
joined.to_csv("9NSF_pd1.csv", index=False)
geneWeights.to_csv("9NSF_pd1_W.csv")